In [70]:
import pandas as pd
import numpy as np
import glob
from sklearn.linear_model import LinearRegression

In [64]:
X = []
Y = []

coalmine_list = glob.glob("./赛题数据划分/训练集/*")

# 每个煤矿
for coalmine_train_path in coalmine_list:
    coalmine_train_day_path_list = glob.glob(coalmine_train_path + "/*") # 每个煤矿不同天数
    coalmine_train_label_df = pd.read_excel("./赛题数据划分/训练集标签/" + coalmine_train_path.split("/")[-1] + ".xlsx")
    
    # 每个煤矿某一天的数据，存在多个文件
    for coalmine_train_day_file_path in coalmine_train_day_path_list:
        coalmine_train_day_files_path = glob.glob(coalmine_train_day_file_path + "/*.csv")

        coalmine_train_day = coalmine_train_day_file_path.split(coalmine_train_path.split("/")[-1])[-1]
        
        dfs = []
        for path in coalmine_train_day_files_path:
            df = pd.read_csv(coalmine_train_day_files_path[0], skiprows=4).reset_index().iloc[:, :2]
            df.columns = ["Wavelength", "Data"] # 每个文件数量不同
            dfs.append([
                df["Data"].mean(), df["Data"].sum(), df["Data"].min(), df["Data"].median(), np.ptp(df["Data"]), df["Data"].std()
            ])

        dfs = np.array(dfs)
        dfs_feat = list(dfs.mean(0)) + list(dfs.max(0)) + list(dfs.std(0))
        X.append(dfs_feat)
        Y.append(coalmine_train_label_df[coalmine_train_label_df["名称"] == coalmine_train_day]["发热量(Q)"].values[0])

In [66]:
np.array(X).shape

(70, 18)

In [71]:
model = LinearRegression()
model.fit(X, Y)

LinearRegression()

In [76]:
X_test = []
X_test_name = []
coalmine_list = glob.glob("./赛题数据划分/测试集/*")

# 每个煤矿
for coalmine_train_path in coalmine_list:
    coalmine_train_day_path_list = glob.glob(coalmine_train_path + "/*") # 每个煤矿不同天数
    
    # 每个煤矿某一天的数据，存在多个文件
    for coalmine_train_day_file_path in coalmine_train_day_path_list:
        coalmine_train_day_files_path = glob.glob(coalmine_train_day_file_path + "/*.csv")
        
        dfs = []
        for path in coalmine_train_day_files_path:
            df = pd.read_csv(coalmine_train_day_files_path[0], skiprows=4).reset_index().iloc[:, :2]
            df.columns = ["Wavelength", "Data"] # 每个文件数量不同
            dfs.append([
                df["Data"].mean(), df["Data"].sum(), df["Data"].min(), df["Data"].median(), np.ptp(df["Data"]), df["Data"].std()
            ])

        dfs = np.array(dfs)
        dfs_feat = list(dfs.mean(0)) + list(dfs.max(0)) + list(dfs.std(0))
        X_test.append(dfs_feat)
        X_test_name.append(coalmine_train_day_file_path.split("/")[-1])

In [82]:
!mkdir submit

In [83]:
pd.DataFrame(
    {"名称": X_test_name, "预测发热量_MJ_KG": model.predict(X_test)}
).to_csv("submit/submit.csv", index=None)

In [ ]:
!zip -r submit.zip submit